# Preprocessing and Feature Extraction

This notebook prepares the **STRESS** session of the Wearable Device Dataset for binary REST-vs-STRESS classification. It applies the dataset-specific recording constraints, creates 30-second windows with 50% overlap, extracts multimodal physiological features, performs quality checks, and exports the cleaned feature table used by the later notebooks.

## 1. Setup

In [4]:
!pip install neurokit2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 11.9 MB/s eta 0:00:00


In [5]:
import warnings
from neurokit2.misc import NeuroKitWarning

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=NeuroKitWarning)

In [6]:
import pandas as pd
import os
import numpy as np
import datetime
import matplotlib.pyplot as plt
import re
from pathlib import Path

import neurokit2 as nk

from scipy.stats import skew, kurtosis
from scipy.stats import linregress

from tqdm import tqdm

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Repository paths

In [18]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_ROOT = Path(
    "/content/drive/MyDrive/"
    "wearable-device-dataset-from-induced-stress-and-structured-exercise-sessions-1.0.1"
)

dataset_path = RAW_DATA_ROOT / "Wearable_Dataset"
stress_level_v1_path = RAW_DATA_ROOT / "Stress_Level_v1.csv"
stress_level_v2_path = RAW_DATA_ROOT / "Stress_Level_v2.csv"

## 2. Dataset Loading Utilities

The following helper functions are adapted from the utilities supplied with the dataset and are used to load the Empatica E4 recordings and their timestamps.

In [19]:
def graph_multiple(signals,timeline,subject_signals,state):

    plt.figure(figsize=(25,15))

    keys = list(signals[subject_signals].keys())
    keys.remove("tags")

    i=1

    for key in keys:
        plt.subplot(len(keys),1,i)
        if i==1:
            plt.title(subject_signals + "  -  "+state)
        if key=='ACC':
            acc=moving_average(signals[subject_signals][key])
            plt.plot(acc,label=key)
        else:
            plt.plot(timeline[subject_signals][key],signals[subject_signals][key],label=key)

        for tag in signals[subject_signals]["tags"][1:]:
            plt.axvline(x=tag, color='r', linestyle='-')

        if state=='STRESS' and signals[subject_signals]["tags"]:
            if 'S' in subject_signals: #first version
                plt.axvspan(signals[subject_signals]["tags"][3], signals[subject_signals]["tags"][4], color='red', alpha=0.2) #stroop
                plt.axvspan(signals[subject_signals]["tags"][5], signals[subject_signals]["tags"][6], color='red', alpha=0.2)# tmct
                plt.axvspan(signals[subject_signals]["tags"][7], signals[subject_signals]["tags"][8], color='red', alpha=0.2)#real opinion
                plt.axvspan(signals[subject_signals]["tags"][9], signals[subject_signals]["tags"][10], color='red', alpha=0.2)#opposite opinion
                plt.axvspan(signals[subject_signals]["tags"][11], signals[subject_signals]["tags"][12], color='red', alpha=0.2)#subtract test

            else: #second version
                plt.axvspan(signals[subject_signals]["tags"][2], signals[subject_signals]["tags"][3], color='red', alpha=0.2) #tmct
                plt.axvspan(signals[subject_signals]["tags"][4], signals[subject_signals]["tags"][5], color='red', alpha=0.2)#real opinion
                plt.axvspan(signals[subject_signals]["tags"][6], signals[subject_signals]["tags"][7], color='red', alpha=0.2)#opposite opinion
                plt.axvspan(signals[subject_signals]["tags"][8], signals[subject_signals]["tags"][9], color='red', alpha=0.2)#subtract test

        plt.legend()
        plt.grid()
        i = i+1
    plt.show()

In [20]:
# create a vector from the data frame (signal imported by pandas)
def create_df_array(dataframe):
    matrix_df=dataframe.values
    # returns 2-d matrix
    matrix = np.array(matrix_df)
    array_df = matrix.flatten()# Convert matrix into an array
    return array_df

# convert UTC arrays to arrays in seconds relative to 0 (record beginning)
def time_abs_(UTC_array):
    new_array=[]
    for utc in UTC_array:
        time=(datetime.datetime.strptime(utc,'%Y-%m-%d %H:%M:%S')-datetime.datetime.strptime(UTC_array[0], '%Y-%m-%d %H:%M:%S')).total_seconds()
        new_array.append(int(time))
    return new_array

In [21]:
def moving_average(acc_data):
    # Initialization of variables
    avg = 0
    prevX, prevY, prevZ = 0, 0, 0
    results = []
    # Each second (32 samples) the acceleration data is summarized using the following method:
    for i in range(0, len(acc_data), 32):
        sum_ = 0
        buffX = acc_data[i:i+32, 0]
        buffY = acc_data[i:i+32, 1]
        buffZ = acc_data[i:i+32, 2]

        for j in range(len(buffX)):
            sum_ += max(
                abs(buffX[j] - prevX),
                abs(buffY[j] - prevY),
                abs(buffZ[j] - prevZ)
            )
            prevX, prevY, prevZ = buffX[j], buffY[j], buffZ[j]
        #The output is then filtered:
        avg = avg * 0.9 + (sum_ / 32) * 0.1 #
        results.append(avg)

    return results

In [22]:
def read_signals(main_folder):
    signal_dict = {}
    time_dict = {}
    fs_dict = {}

    # Get a list of subfolders in the main folder
    subfolders = next(os.walk(main_folder))[1]

    utc_start_dict={}
    for folder_name in subfolders:
            csv_path = f'{main_folder}/{folder_name}/EDA.csv'
            df=pd.read_csv(csv_path)
            utc_start_dict[folder_name]= df.columns.tolist()

    # Iterate over the subfolders
    for folder_name in subfolders:
        folder_path = os.path.join(main_folder, folder_name)
        # Get a list of files in the subfolder
        files = os.listdir(folder_path)

        # Initialize a dictionary for the signals in the current subfolder
        signals = {}
        time_line = {}
        fs_signal= {}

        # Define the list of desired file names
        desired_files = ['EDA.csv', 'BVP.csv', 'HR.csv', 'TEMP.csv','tags.csv','ACC.csv']

        # Iterate over the files in the subfolder
        for file_name in files:
            file_path = os.path.join(folder_path, file_name)

            # Check if it's a CSV file and if it is in the desired files list
            if file_name.endswith('.csv') and file_name in desired_files:
                # Read the CSV file and store the signal data

                # if file_name == 'tags.csv':
                #     try:
                #         df = pd.read_csv(file_path,header=None)
                #         tags_vector = create_df_array(df)
                #         tags_UTC_vector =np.insert(tags_vector,0,utc_start_dict[folder_name])
                #         signal_array= time_abs_(tags_UTC_vector)
                #     except pd.errors.EmptyDataError:
                #         signal_array=[]
                if file_name == 'tags.csv':
                    try:
                        df = pd.read_csv(file_path, header=None)
                        tags_vector = create_df_array(df)
                        tags_UTC_vector = np.insert(tags_vector, 0, utc_start_dict[folder_name])
                        signal_array = time_abs_(tags_UTC_vector)

                        time_array = signal_array   # or appropriate transformation
                        fs = None  # or 0, since tags don't have sampling frequency

                    except pd.errors.EmptyDataError:
                        signal_array = []
                        time_array = []
                        fs = None

                else:
                    df = pd.read_csv(file_path)
                    fs = int(df.iloc[0, 0]) #Get sampling frequency
                    # fs= df.loc[0]
                    # fs=int(fs[0])
                    df.drop([0],axis = 0,inplace=True)
                    signal_array = df.values
                    time_array = np.linspace(0, len(signal_array)/fs,len(signal_array))

                signal_name = file_name.split('.')[0]
                signals[signal_name] = signal_array
                time_line[signal_name] = time_array
                fs_signal[signal_name] = fs

        # Store the signals of the current subfolder in the main dictionary
        signal_dict[folder_name] = signals
        time_dict[folder_name] = time_line
        fs_dict[folder_name] = fs_signal

    return signal_dict, time_dict, fs_dict


## 3. Load the STRESS Session

The original dataset contains AEROBIC, ANAEROBIC, and STRESS sessions. This project uses only the **STRESS** protocol because it contains the labelled resting and induced-stress stages required for the binary classification task.

In [23]:
states=os.listdir(dataset_path) #['AEROBIC', 'ANAEROBIC', 'STRESS']
print(states)

signal_data={}
time_data={}
fs_dict={}
participants={}

# Returns three dictionaries with subjects info: raw signals (signal_data),
# temporal data ready to graph (time_data) and sample frequency for escha signal(fs_dict).
for state in states:
    if state == '.DS_Store':
      continue
    folder_path = f'{dataset_path}/{state}'
    participants[state]=os.listdir(folder_path)
    signal_data[state], time_data[state], fs_dict[state] = read_signals(folder_path)

['STRESS', 'ANAEROBIC', 'AEROBIC']


In [24]:
STATE_TO_USE = "STRESS"

stress_signals = signal_data[STATE_TO_USE]
stress_time = time_data[STATE_TO_USE]
stress_fs = fs_dict[STATE_TO_USE]

print("Number of STRESS recordings:", len(stress_signals))
print(sorted(stress_signals.keys()))

Number of STRESS recordings: 37
['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15', 'S16', 'S17', 'S18', 'f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13', 'f14_a', 'f14_b', 'f15', 'f16', 'f17', 'f18']


## 4. Dataset-Specific Quality Constraints

The STRESS recordings contain a small number of documented signal-quality issues that must be handled before windowing. `S02` is trimmed before duplicated values begin, `f07` is excluded from the multimodal dataset because PPG and temperature were obstructed, and the two `f14` recordings are treated as one participant.

In [25]:
S02_CUTOFF_SEC = 1548  # duplicated values start around this time

EXCLUDE_FROM_MAIN = {
    "f07": "Invalid PPG and TEMP because protection dock covered sensors."
}

SPECIAL_CASES = {
    "S02": "Trim windows ending after 1548 seconds because duplicated raw values begin there.",
    "f14": "f14_a and f14_b are separate recordings belonging to the same participant."
}

### 4.1 Recording and participant identifiers

In [26]:
def get_participant_id(recording_id):
    """
    Converts recording folder names into participant IDs.

    Examples:
    S02   -> S02
    f07   -> f07
    f14_a -> f14
    f14_b -> f14
    """
    match = re.match(r"^(S\d+|f\d+)", recording_id)
    if match is None:
        return recording_id
    return match.group(1)


def keep_window(participant_id, recording_id, end_sec):
    """
    Applies dataset-level exclusion/trimming rules.
    """
    if participant_id in EXCLUDE_FROM_MAIN:
        return False

    if participant_id == "S02" and end_sec > S02_CUTOFF_SEC:
        return False

    return True

In [27]:
for rid in ["S02", "f07", "f14_a", "f14_b"]:
    print(rid, get_participant_id(rid))

S02 S02
f07 f07
f14_a f14
f14_b f14


In [28]:
recording_rows = []

for recording_id, signals in stress_signals.items():
    participant_id = get_participant_id(recording_id)

    row = {
        "recording_id": recording_id,
        "participant_id": participant_id,
        "has_BVP": "BVP" in signals,
        "has_EDA": "EDA" in signals,
        "has_ACC": "ACC" in signals,
        "has_TEMP": "TEMP" in signals,
        "has_tags": "tags" in signals,
    }

    for sig in ["BVP", "EDA", "ACC", "TEMP"]:
        if sig in signals:
            row[f"{sig}_shape"] = signals[sig].shape
            row[f"{sig}_fs"] = stress_fs[recording_id].get(sig, None)
        else:
            row[f"{sig}_shape"] = None
            row[f"{sig}_fs"] = None

    recording_rows.append(row)

In [29]:
recordings_df = pd.DataFrame(recording_rows)
recordings_df = recordings_df.sort_values(["participant_id", "recording_id"]).reset_index(drop=True)

recordings_df

,recording_id,participant_id,has_BVP,has_EDA,has_ACC,has_TEMP,has_tags,BVP_shape,BVP_fs,EDA_shape,EDA_fs,ACC_shape,ACC_fs,TEMP_shape,TEMP_fs
0,S01,S01,True,True,True,True,True,"(142351, 1)",64,"(8898, 1)",4,"(71178, 3)",32,"(8896, 1)",4
1,S02,S02,True,True,True,True,True,"(198176, 1)",64,"(12384, 1)",4,"(99084, 3)",32,"(12384, 1)",4
2,S03,S03,True,True,True,True,True,"(101486, 1)",64,"(6342, 1)",4,"(50748, 3)",32,"(6344, 1)",4
3,S04,S04,True,True,True,True,True,"(110924, 1)",64,"(6930, 1)",4,"(55458, 3)",32,"(6928, 1)",4
4,S05,S05,True,True,True,True,True,"(101178, 1)",64,"(6324, 1)",4,"(50592, 3)",32,"(6320, 1)",4
5,S06,S06,True,True,True,True,True,"(92400, 1)",64,"(5772, 1)",4,"(46200, 3)",32,"(5776, 1)",4
6,S07,S07,True,True,True,True,True,"(91432, 1)",64,"(5712, 1)",4,"(45714, 3)",32,"(5712, 1)",4
7,S08,S08,True,True,True,True,True,"(91443, 1)",64,"(5712, 1)",4,"(45720, 3)",32,"(5712, 1)",4
8,S09,S09,True,True,True,True,True,"(92708, 1)",64,"(5790, 1)",4,"(46356, 3)",32,"(5792, 1)",4
9,S10,S10,True,True,True,True,True,"(106128, 1)",64,"(6630, 1)",4,"(53064, 3)",32,"(6632, 1)",4


In [30]:
recordings_df[
    recordings_df["participant_id"].isin(["S02", "f07", "f14"])
]

,recording_id,participant_id,has_BVP,has_EDA,has_ACC,has_TEMP,has_tags,BVP_shape,BVP_fs,EDA_shape,EDA_fs,ACC_shape,ACC_fs,TEMP_shape,TEMP_fs
1,S02,S02,True,True,True,True,True,"(198176, 1)",64,"(12384, 1)",4,"(99084, 3)",32,"(12384, 1)",4
24,f07,f07,True,True,True,True,True,"(221342, 1)",64,"(13824, 1)",4,"(110652, 3)",32,"(13824, 1)",4
31,f14_a,f14,True,True,True,True,True,"(45034, 1)",64,"(2814, 1)",4,"(22518, 3)",32,"(2816, 1)",4
32,f14_b,f14,True,True,True,True,True,"(138765, 1)",64,"(8682, 1)",4,"(69402, 3)",32,"(8672, 1)",4


### 4.2 Inspect protocol tag intervals

In [31]:
def inspect_tag_intervals(recording_id):
    tags = stress_signals[recording_id].get("tags", [])

    if tags is None or len(tags) < 2:
        return pd.DataFrame()

    tags = np.asarray(tags, dtype=float)
    participant_id = get_participant_id(recording_id)

    rows = []

    for block_id in range(len(tags) - 1):
        start_sec = tags[block_id]
        end_sec = tags[block_id + 1]

        rows.append({
            "recording_id": recording_id,
            "participant_id": participant_id,
            "block_id": block_id,
            "start_sec": start_sec,
            "end_sec": end_sec,
            "duration_sec": end_sec - start_sec,
        })

    return pd.DataFrame(rows)


all_intervals = []

for recording_id in stress_signals.keys():
    df_intervals = inspect_tag_intervals(recording_id)
    if not df_intervals.empty:
        all_intervals.append(df_intervals)

intervals_df = pd.concat(all_intervals, ignore_index=True)
intervals_df = intervals_df.sort_values(["participant_id", "recording_id", "block_id"]).reset_index(drop=True)

intervals_df.head(20)

,recording_id,participant_id,block_id,start_sec,end_sec,duration_sec
0,S01,S01,0,0.0,371.0,371.0
1,S01,S01,1,371.0,618.0,247.0
2,S01,S01,2,618.0,793.0,175.0
3,S01,S01,3,793.0,898.0,105.0
4,S01,S01,4,898.0,1338.0,440.0
5,S01,S01,5,1338.0,1463.0,125.0
6,S01,S01,6,1463.0,1770.0,307.0
7,S01,S01,7,1770.0,1798.0,28.0
8,S01,S01,8,1798.0,1836.0,38.0
9,S01,S01,9,1836.0,1873.0,37.0


In [32]:
intervals_df[
    intervals_df["recording_id"].isin(["S02", "f07", "f14_a"])
]

,recording_id,participant_id,block_id,start_sec,end_sec,duration_sec
13,S02,S02,0,0.0,1.0,1.0
14,S02,S02,1,1.0,193.0,192.0
15,S02,S02,2,193.0,279.0,86.0
16,S02,S02,3,279.0,395.0,116.0
17,S02,S02,4,395.0,644.0,249.0
18,S02,S02,5,644.0,771.0,127.0
19,S02,S02,6,771.0,1126.0,355.0
20,S02,S02,7,1126.0,1161.0,35.0
21,S02,S02,8,1161.0,1189.0,28.0
22,S02,S02,9,1189.0,1223.0,34.0


## 5. Protocol Labels

Protocol versions 1 and 2 use different tag positions. The cells below map each tagged block to a protocol-stage label and then to the binary target (`REST` or `STRESS`). `f14_a` is handled as the participant's baseline recording.

In [33]:
# ADD MANUAL BASELINE BLOCK FOR f14_a

def get_recording_duration_sec(recording_id):
    """
    Returns the shortest available modality duration for a recording.
    This avoids creating windows longer than one of the signals.
    """
    signals = stress_signals[recording_id]
    fs_info = stress_fs[recording_id]

    durations = []

    for sig in ["BVP", "EDA", "ACC", "TEMP"]:
        if sig in signals:
            fs = fs_info[sig]
            durations.append(len(signals[sig]) / fs)

    return min(durations)


# Add f14_a only if it exists and is not already in intervals_df
if "f14_a" in stress_signals and "f14_a" not in intervals_df["recording_id"].unique():
    f14a_duration = get_recording_duration_sec("f14_a")

    f14a_row = {
        "recording_id": "f14_a",
        "participant_id": "f14",
        "block_id": -1,
        "start_sec": 0.0,
        "end_sec": f14a_duration,
        "duration_sec": f14a_duration,
        "stage_label": "baseline_rest",
        "binary_label": "REST",
    }

    intervals_df = pd.concat(
        [intervals_df, pd.DataFrame([f14a_row])],
        ignore_index=True
    )

intervals_df = intervals_df.sort_values(
    ["participant_id", "recording_id", "block_id"]
).reset_index(drop=True)

intervals_df[intervals_df["participant_id"] == "f14"]

,recording_id,participant_id,block_id,start_sec,end_sec,duration_sec,stage_label,binary_label
351,f14_a,f14,-1,0.0,703.5,703.5,baseline_rest,REST
352,f14_b,f14,0,0.0,142.0,142.0,NaN,NaN
353,f14_b,f14,1,142.0,232.0,90.0,NaN,NaN
354,f14_b,f14,2,232.0,478.0,246.0,NaN,NaN
355,f14_b,f14,3,478.0,1189.0,711.0,NaN,NaN
356,f14_b,f14,4,1189.0,1228.0,39.0,NaN,NaN
357,f14_b,f14,5,1228.0,1270.0,42.0,NaN,NaN
358,f14_b,f14,6,1270.0,1319.0,49.0,NaN,NaN
359,f14_b,f14,7,1319.0,1958.0,639.0,NaN,NaN
360,f14_b,f14,8,1958.0,2001.0,43.0,NaN,NaN


In [34]:
V1_STRESS_BLOCKS = {
    3: "stroop",
    5: "tmct",
    7: "real_opinion",
    9: "opposite_opinion",
    11: "subtraction",
}

V2_STRESS_BLOCKS = {
    2: "tmct",
    4: "real_opinion",
    6: "opposite_opinion",
    8: "subtraction",
}

V1_REST_BLOCKS = {
    1: "baseline_rest",
    4: "post_stroop_rest",
    6: "post_tmct_rest",
    8: "post_real_opinion_rest",
    10: "post_opposite_opinion_rest",
}

V2_REST_BLOCKS = {
    0: "baseline_rest",
    1: "pre_task_rest",
    3: "post_tmct_rest",
    5: "post_real_opinion_rest",
    7: "post_opposite_opinion_rest",
}

In [35]:
def get_protocol_version(participant_id):
    if participant_id.startswith("S"):
        return "v1"
    elif participant_id.startswith("f"):
        return "v2"
    return "unknown"


def label_block(recording_id, participant_id, block_id):
    # Manual f14_a baseline case
    if recording_id == "f14_a":
        return "baseline_rest", "REST"

    version = get_protocol_version(participant_id)

    if version == "v1":
        if block_id in V1_STRESS_BLOCKS:
            return V1_STRESS_BLOCKS[block_id], "STRESS"
        elif block_id in V1_REST_BLOCKS:
            return V1_REST_BLOCKS[block_id], "REST"
        else:
            return "ignore", "IGNORE"

    elif version == "v2":
        if block_id in V2_STRESS_BLOCKS:
            return V2_STRESS_BLOCKS[block_id], "STRESS"
        elif block_id in V2_REST_BLOCKS:
            return V2_REST_BLOCKS[block_id], "REST"
        else:
            return "ignore", "IGNORE"

    return "ignore", "IGNORE"


# Apply labels only to rows that do not already have manual labels
intervals_df[["stage_label", "binary_label"]] = intervals_df.apply(
    lambda row: pd.Series(
        label_block(
            row["recording_id"],
            row["participant_id"],
            row["block_id"]
        )
    ),
    axis=1
)

intervals_df.head(30)

,recording_id,participant_id,block_id,start_sec,end_sec,duration_sec,stage_label,binary_label
0,S01,S01,0,0.0,371.0,371.0,ignore,IGNORE
1,S01,S01,1,371.0,618.0,247.0,baseline_rest,REST
2,S01,S01,2,618.0,793.0,175.0,ignore,IGNORE
3,S01,S01,3,793.0,898.0,105.0,stroop,STRESS
4,S01,S01,4,898.0,1338.0,440.0,post_stroop_rest,REST
5,S01,S01,5,1338.0,1463.0,125.0,tmct,STRESS
6,S01,S01,6,1463.0,1770.0,307.0,post_tmct_rest,REST
7,S01,S01,7,1770.0,1798.0,28.0,real_opinion,STRESS
8,S01,S01,8,1798.0,1836.0,38.0,post_real_opinion_rest,REST
9,S01,S01,9,1836.0,1873.0,37.0,opposite_opinion,STRESS


In [36]:
intervals_df[
    intervals_df["participant_id"].isin(["S02", "f07", "f14"])
][
    ["recording_id", "participant_id", "block_id", "start_sec",
     "end_sec", "duration_sec", "stage_label", "binary_label"]
]

,recording_id,participant_id,block_id,start_sec,end_sec,duration_sec,stage_label,binary_label
13,S02,S02,0,0.0,1.0,1.0,ignore,IGNORE
14,S02,S02,1,1.0,193.0,192.0,baseline_rest,REST
15,S02,S02,2,193.0,279.0,86.0,ignore,IGNORE
16,S02,S02,3,279.0,395.0,116.0,stroop,STRESS
17,S02,S02,4,395.0,644.0,249.0,post_stroop_rest,REST
18,S02,S02,5,644.0,771.0,127.0,tmct,STRESS
19,S02,S02,6,771.0,1126.0,355.0,post_tmct_rest,REST
20,S02,S02,7,1126.0,1161.0,35.0,real_opinion,STRESS
21,S02,S02,8,1161.0,1189.0,28.0,post_real_opinion_rest,REST
22,S02,S02,9,1189.0,1223.0,34.0,opposite_opinion,STRESS


In [37]:
print(intervals_df["binary_label"].value_counts())

intervals_df.groupby(["participant_id", "binary_label"]).size().unstack(fill_value=0).head(25)

binary_label
REST      181
STRESS    162
IGNORE     54
Name: count, dtype: int64


binary_label,IGNORE,REST,STRESS
participant_id,,,
S01,3,5,5
S02,3,5,5
S03,3,5,5
S04,3,5,5
S05,3,5,5
S06,3,5,5
S07,3,5,5
S08,3,5,5
S09,3,5,5


In [38]:
short_blocks = intervals_df[
    (intervals_df["binary_label"] != "IGNORE") &
    (intervals_df["duration_sec"] < 30)
][
    ["recording_id", "participant_id", "block_id",
     "duration_sec", "stage_label", "binary_label"]
]

short_blocks

,recording_id,participant_id,block_id,duration_sec,stage_label,binary_label
7,S01,S01,7,28.0,real_opinion,STRESS
21,S02,S02,8,28.0,post_real_opinion_rest,REST
23,S02,S02,10,24.0,post_opposite_opinion_rest,REST
34,S03,S03,8,19.0,post_real_opinion_rest,REST
74,S06,S06,9,28.0,opposite_opinion,STRESS
75,S06,S06,10,28.0,post_opposite_opinion_rest,REST
99,S08,S08,8,26.0,post_real_opinion_rest,REST
124,S10,S10,7,29.0,real_opinion,STRESS
126,S10,S10,9,29.0,opposite_opinion,STRESS
127,S10,S10,10,27.0,post_opposite_opinion_rest,REST


## 6. Window Generation

Each labelled block is segmented into **30-second windows with 50% overlap** (15-second step). Windows shorter than 30 seconds, ignored protocol blocks, excluded participants and the corrupted tail of `S02` are not retained.

In [39]:
WINDOW_SEC = 30
OVERLAP = 0.5
STEP_SEC = int(WINDOW_SEC * (1 - OVERLAP))  # 15 seconds

def create_windows_from_block(row):
    recording_id = row["recording_id"]
    participant_id = row["participant_id"]
    block_id = row["block_id"]
    stage_label = row["stage_label"]
    binary_label = row["binary_label"]

    block_start = float(row["start_sec"])
    block_end = float(row["end_sec"])
    block_duration = block_end - block_start

    windows = []

    if binary_label == "IGNORE":
        return windows

    # Strict 30s windows only
    if block_duration < WINDOW_SEC:
        return windows

    current_start = block_start

    while current_start + WINDOW_SEC <= block_end:
        current_end = current_start + WINDOW_SEC

        if keep_window(participant_id, recording_id, current_end):
            windows.append({
                "recording_id": recording_id,
                "participant_id": participant_id,
                "block_id": block_id,
                "stage_label": stage_label,
                "label": binary_label,
                "start_sec": current_start,
                "end_sec": current_end,
                "window_sec": WINDOW_SEC,
            })

        current_start += STEP_SEC

    return windows


window_rows = []

for _, row in intervals_df.iterrows():
    window_rows.extend(create_windows_from_block(row))

windows_df = pd.DataFrame(window_rows)

windows_df = windows_df.sort_values(
    ["participant_id", "recording_id", "start_sec"]
).reset_index(drop=True)

print("Windows shape:", windows_df.shape)
print(windows_df["label"].value_counts())
windows_df.head()

Windows shape: (4560, 8)
label
REST      3919
STRESS     641
Name: count, dtype: int64


,recording_id,participant_id,block_id,stage_label,label,start_sec,end_sec,window_sec
0,S01,S01,1,baseline_rest,REST,371.0,401.0,30
1,S01,S01,1,baseline_rest,REST,386.0,416.0,30
2,S01,S01,1,baseline_rest,REST,401.0,431.0,30
3,S01,S01,1,baseline_rest,REST,416.0,446.0,30
4,S01,S01,1,baseline_rest,REST,431.0,461.0,30


In [40]:
# f14_a is the real baseline, f14_b block 0 is only rest/pre-task
windows_df.loc[
    (windows_df["recording_id"] == "f14_b") & (windows_df["block_id"] == 0),
    "stage_label"
] = "pre_task_rest"

### 6.1 Window-level sanity checks

In [41]:
# f07 should be absent
print("f07 windows:")
display(windows_df[windows_df["participant_id"] == "f07"])

# S02 should have no windows ending after 1548s
print("S02 max end_sec:")
display(
    windows_df[windows_df["participant_id"] == "S02"]
    [["recording_id", "start_sec", "end_sec", "stage_label", "label"]]
    .tail()
)

# f14 should include f14_a and f14_b under the same participant_id
print("f14 windows:")
display(
    windows_df[windows_df["participant_id"] == "f14"]
    [["recording_id", "participant_id", "start_sec", "end_sec", "stage_label", "label"]]
    .head(30)
)

display(
    windows_df[windows_df["participant_id"] == "f14"]
    .groupby(["recording_id", "label", "stage_label"])
    .size()
    .reset_index(name="n_windows")
)

f07 windows:


,recording_id,participant_id,block_id,stage_label,label,start_sec,end_sec,window_sec


S02 max end_sec:


,recording_id,start_sec,end_sec,stage_label,label
138,S02,1071.0,1101.0,post_tmct_rest,REST
139,S02,1086.0,1116.0,post_tmct_rest,REST
140,S02,1126.0,1156.0,real_opinion,STRESS
141,S02,1189.0,1219.0,opposite_opinion,STRESS
142,S02,1247.0,1277.0,subtraction,STRESS


f14 windows:


,recording_id,participant_id,start_sec,end_sec,stage_label,label
3603,f14_a,f14,0.0,30.0,baseline_rest,REST
3604,f14_a,f14,15.0,45.0,baseline_rest,REST
3605,f14_a,f14,30.0,60.0,baseline_rest,REST
3606,f14_a,f14,45.0,75.0,baseline_rest,REST
3607,f14_a,f14,60.0,90.0,baseline_rest,REST
3608,f14_a,f14,75.0,105.0,baseline_rest,REST
3609,f14_a,f14,90.0,120.0,baseline_rest,REST
3610,f14_a,f14,105.0,135.0,baseline_rest,REST
3611,f14_a,f14,120.0,150.0,baseline_rest,REST
3612,f14_a,f14,135.0,165.0,baseline_rest,REST


,recording_id,label,stage_label,n_windows
0,f14_a,REST,baseline_rest,45
1,f14_b,REST,post_opposite_opinion_rest,41
2,f14_b,REST,post_real_opinion_rest,1
3,f14_b,REST,post_tmct_rest,46
4,f14_b,REST,pre_task_rest,13
5,f14_b,STRESS,opposite_opinion,2
6,f14_b,STRESS,real_opinion,1
7,f14_b,STRESS,subtraction,1
8,f14_b,STRESS,tmct,15


In [42]:
windows_df.to_csv(DATA_DIR / "stress_window_metadata_30s.csv", index=False)

## 7. Feature Extraction

Features are extracted separately for PPG/BVP, EDA, skin temperature, and accelerometer signals. A window is retained only when all required modalities pass the corresponding extraction and quality checks.

In [43]:
def get_signal_window(recording_id, signal_name, start_sec, end_sec):
    """
    Extracts one signal segment for one row from windows_df.
    """
    signal = stress_signals[recording_id][signal_name]
    fs = stress_fs[recording_id][signal_name]

    start_sample = int(round(start_sec * fs))
    end_sample = int(round(end_sec * fs))

    segment = signal[start_sample:end_sample]

    if signal_name in ["BVP", "EDA", "TEMP"]:
        segment = segment.flatten()

    return segment, fs

In [44]:
def safe_skew(x):
    if len(x) < 3 or np.std(x) == 0:
        return 0.0
    return skew(x, nan_policy="omit")


def safe_kurtosis(x):
    if len(x) < 4 or np.std(x) == 0:
        return 0.0
    return kurtosis(x, nan_policy="omit")


def basic_stats(x, prefix):
    x = np.asarray(x, dtype=float)

    return {
        f"{prefix}_Mean": np.mean(x),
        f"{prefix}_Median": np.median(x),
        f"{prefix}_SD": np.std(x),
        f"{prefix}_Min": np.min(x),
        f"{prefix}_Max": np.max(x),
        f"{prefix}_Range": np.max(x) - np.min(x),
        f"{prefix}_Skew": safe_skew(x),
        f"{prefix}_Kurtosis": safe_kurtosis(x),
    }

### 7.1 PPG/BVP and HRV features

In [45]:
PPG_QUALITY_THRESHOLD = 0.50
MIN_PPG_PEAKS = 20

HRV_SELECTED_COLUMNS = [
    # time-domain
    "HRV_MeanNN",
    "HRV_SDNN",
    "HRV_RMSSD",
    "HRV_SDSD",
    "HRV_CVNN",
    "HRV_CVSD",
    "HRV_pNN20",
    "HRV_pNN50",

    # frequency-domain, less stable in 30s windows
    "HRV_LF",
    "HRV_HF",
    "HRV_LFHF",

    # nonlinear
    "HRV_SD1",
    "HRV_SD2",
    "HRV_SD1SD2",
    "HRV_SampEn",
    "HRV_ApEn",
]


def extract_bvp_features(bvp_segment, fs):
    bvp_segment = np.asarray(bvp_segment, dtype=float)

    expected_len = int(30 * fs)
    if len(bvp_segment) != expected_len:
        return None, "wrong_bvp_length"

    if np.std(bvp_segment) < 0.01:
        return None, "flat_bvp"

    if np.mean(bvp_segment == 0) > 0.30:
        return None, "too_many_bvp_zeros"

    try:
        # filtering style as recommended for the dataset: 0.5–10 Hz bandpass
        bvp_clean = nk.signal_filter(
            bvp_segment,
            sampling_rate=fs,
            lowcut=0.5,
            highcut=10.0,
            method="chebyshevii",
            order=4
        )

        processed, info = nk.ppg_process(bvp_clean, sampling_rate=fs)

        quality = processed["PPG_Quality"].mean()
        peaks = np.asarray(info["PPG_Peaks"])

        if quality < PPG_QUALITY_THRESHOLD:
            return None, "low_ppg_quality"

        if len(peaks) < MIN_PPG_PEAKS:
            return None, "too_few_ppg_peaks"

        features = {}

        # raw/clean BVP statistics
        features.update(basic_stats(bvp_clean, "BVP_Clean"))

        # derivative features
        d1 = np.diff(bvp_clean)
        d2 = np.diff(d1)

        features.update({
            "BVP_Deriv1_Mean": np.mean(d1),
            "BVP_Deriv1_SD": np.std(d1),
            "BVP_Deriv2_Mean": np.mean(d2),
            "BVP_Deriv2_SD": np.std(d2),
        })

        # PPG rate features
        ppg_rate = processed["PPG_Rate"].dropna()

        features.update({
            "PPG_Rate_Mean": ppg_rate.mean(),
            "PPG_Rate_SD": ppg_rate.std(),
            "PPG_Rate_Min": ppg_rate.min(),
            "PPG_Rate_Max": ppg_rate.max(),
            "PPG_Quality_Mean": quality,
            "PPG_Peaks_N": len(peaks),
        })

        # pulse-to-pulse interval features
        ppi = np.diff(peaks) / fs  # seconds

        features.update({
            "PPI_Mean": np.mean(ppi),
            "PPI_Median": np.median(ppi),
            "PPI_SD": np.std(ppi),
            "PPI_Min": np.min(ppi),
            "PPI_Max": np.max(ppi),
        })

        # HRV features from PPG peaks
        hrv_df = nk.hrv(peaks, sampling_rate=fs, show=False)

        for col in HRV_SELECTED_COLUMNS:
            if col in hrv_df.columns:
                features[col] = hrv_df[col].iloc[0]
            else:
                features[col] = np.nan

        return features, None

    except Exception as e:
        return None, f"bvp_error: {e}"

### 7.2 EDA features

In [46]:
def extract_eda_features(eda_segment, fs):
    eda_segment = np.asarray(eda_segment, dtype=float)

    expected_len = int(30 * fs)
    if len(eda_segment) != expected_len:
        return None, "wrong_eda_length"

    if np.mean(eda_segment == 0) > 0.30:
        return None, "too_many_eda_zeros"

    try:
        # low-pass filtering for Empatica EDA at 4 Hz.
        # cutoff must be below Nyquist frequency.
        highcut = min(1.99, (fs / 2) * 0.99)

        eda_clean = nk.signal_filter(
            eda_segment,
            sampling_rate=fs,
            highcut=highcut,
            method="butterworth",
            order=5
        )

        signals, info = nk.eda_process(eda_clean, sampling_rate=fs)

        tonic = signals["EDA_Tonic"].values
        phasic = signals["EDA_Phasic"].values

        features = {}

        # raw and decomposed EDA statistics
        features.update(basic_stats(eda_segment, "EDA_Raw"))
        features.update(basic_stats(tonic, "EDA_Tonic"))
        features.update(basic_stats(phasic, "EDA_Phasic"))

        # tonic slope
        x = np.arange(len(tonic))
        features["EDA_Tonic_Slope"] = linregress(x, tonic).slope

        # phasic area
        features["EDA_Phasic_AUC"] = np.trapz(phasic, dx=1 / fs)

        # SCR features
        scr_peaks_n = int(signals["SCR_Peaks"].sum())
        scr_amp = signals.loc[signals["SCR_Amplitude"] > 0, "SCR_Amplitude"]

        features.update({
            "SCR_Peaks_N": scr_peaks_n,
            "SCR_Peaks_Per_Min": scr_peaks_n * 2,  # 30s window -> multiply by 2
            "SCR_Presence": int(scr_peaks_n > 0),
            "SCR_Amplitude_Mean": scr_amp.mean() if len(scr_amp) > 0 else 0.0,
            "SCR_Amplitude_Max": scr_amp.max() if len(scr_amp) > 0 else 0.0,
            "SCR_Amplitude_Sum": scr_amp.sum() if len(scr_amp) > 0 else 0.0,
        })

        if "SCR_Height" in signals.columns:
            scr_height = signals.loc[signals["SCR_Height"] > 0, "SCR_Height"]
            features["SCR_Height_Mean"] = scr_height.mean() if len(scr_height) > 0 else 0.0
            features["SCR_Height_Max"] = scr_height.max() if len(scr_height) > 0 else 0.0

        else:
            features["SCR_Height_Mean"] = np.nan
            features["SCR_Height_Max"] = np.nan

        if "SCR_RiseTime" in signals.columns:
            scr_rise = signals.loc[signals["SCR_RiseTime"] > 0, "SCR_RiseTime"]
            features["SCR_RiseTime_Mean"] = scr_rise.mean() if len(scr_rise) > 0 else 0.0
        else:
            features["SCR_RiseTime_Mean"] = np.nan

        if "SCR_RecoveryTime" in signals.columns:
            scr_recovery = signals.loc[signals["SCR_RecoveryTime"] > 0, "SCR_RecoveryTime"]
            features["SCR_RecoveryTime_Mean"] = scr_recovery.mean() if len(scr_recovery) > 0 else 0.0
        else:
            features["SCR_RecoveryTime_Mean"] = np.nan

        return features, None

    except Exception as e:
        return None, f"eda_error: {e}"

### 7.3 Temperature features

In [47]:
def extract_temp_features(temp_segment, fs):
    temp_segment = np.asarray(temp_segment, dtype=float)

    expected_len = int(30 * fs)
    if len(temp_segment) != expected_len:
        return None, "wrong_temp_length"

    if np.mean(temp_segment == 0) > 0.30:
        return None, "too_many_temp_zeros"

    try:
        features = {}

        features.update(basic_stats(temp_segment, "TEMP"))

        x = np.arange(len(temp_segment))
        features["TEMP_Slope"] = linregress(x, temp_segment).slope

        dtemp = np.diff(temp_segment)

        features.update({
            "TEMP_FirstDiff_Mean": np.mean(dtemp),
            "TEMP_FirstDiff_SD": np.std(dtemp),
        })

        return features, None

    except Exception as e:
        return None, f"temp_error: {e}"

### 7.4 Accelerometer features

In [48]:
def extract_acc_features(acc_segment, fs):
    acc_segment = np.asarray(acc_segment, dtype=float)

    expected_len = int(30 * fs)
    if acc_segment.shape[0] != expected_len:
        return None, "wrong_acc_length"

    if acc_segment.ndim != 2 or acc_segment.shape[1] != 3:
        return None, "wrong_acc_shape"

    try:
        # Empatica ACC is usually stored in 1/64g units
        acc_g = acc_segment / 64.0

        x = acc_g[:, 0]
        y = acc_g[:, 1]
        z = acc_g[:, 2]

        mag = np.sqrt(x**2 + y**2 + z**2)
        jerk = np.diff(mag) * fs

        features = {}

        features.update(basic_stats(x, "ACC_X"))
        features.update(basic_stats(y, "ACC_Y"))
        features.update(basic_stats(z, "ACC_Z"))
        features.update(basic_stats(mag, "ACC_Mag"))

        features.update({
            "ACC_SMA": np.mean(np.abs(x) + np.abs(y) + np.abs(z)),
            "ACC_RMS": np.sqrt(np.mean(mag**2)),
            "ACC_Energy": np.sum(mag**2) / len(mag),

            "ACC_Jerk_Mean": np.mean(jerk),
            "ACC_Jerk_SD": np.std(jerk),
            "ACC_Jerk_Min": np.min(jerk),
            "ACC_Jerk_Max": np.max(jerk),

            "ACC_Mag_P05": np.percentile(mag, 5),
            "ACC_Mag_P25": np.percentile(mag, 25),
            "ACC_Mag_P75": np.percentile(mag, 75),
            "ACC_Mag_P95": np.percentile(mag, 95),
        })

        return features, None

    except Exception as e:
        return None, f"acc_error: {e}"

### 7.5 Extract features for all valid windows

In [49]:
feature_rows = []
failed_rows = []

for idx, row in tqdm(windows_df.iterrows(), total=len(windows_df), desc="Extracting features"):
    recording_id = row["recording_id"]
    participant_id = row["participant_id"]

    meta = {
        "recording_id": recording_id,
        "participant_id": participant_id,
        "block_id": row["block_id"],
        "stage_label": row["stage_label"],
        "label": row["label"],
        "start_sec": row["start_sec"],
        "end_sec": row["end_sec"],
        "window_sec": row["window_sec"],
    }

    try:
        bvp_segment, fs_bvp = get_signal_window(recording_id, "BVP", row["start_sec"], row["end_sec"])
        eda_segment, fs_eda = get_signal_window(recording_id, "EDA", row["start_sec"], row["end_sec"])
        temp_segment, fs_temp = get_signal_window(recording_id, "TEMP", row["start_sec"], row["end_sec"])
        acc_segment, fs_acc = get_signal_window(recording_id, "ACC", row["start_sec"], row["end_sec"])

        bvp_features, bvp_error = extract_bvp_features(bvp_segment, fs_bvp)
        eda_features, eda_error = extract_eda_features(eda_segment, fs_eda)
        temp_features, temp_error = extract_temp_features(temp_segment, fs_temp)
        acc_features, acc_error = extract_acc_features(acc_segment, fs_acc)

        errors = {
            "bvp_error": bvp_error,
            "eda_error": eda_error,
            "temp_error": temp_error,
            "acc_error": acc_error,
        }

        # for the main multimodal dataset, i will only keep windows where all modalities are valid
        if any(error is not None for error in errors.values()):
            failed_rows.append({**meta, **errors})
            continue

        feature_row = {
            **meta,
            **bvp_features,
            **eda_features,
            **temp_features,
            **acc_features,
        }

        feature_rows.append(feature_row)

    except Exception as e:
        failed_rows.append({
            **meta,
            "bvp_error": None,
            "eda_error": None,
            "temp_error": None,
            "acc_error": None,
            "general_error": str(e)
        })


features_df = pd.DataFrame(feature_rows)
failed_df = pd.DataFrame(failed_rows)

print("Valid feature rows:", features_df.shape)
print("Failed rows:", failed_df.shape)

print("\nLabel counts:")
print(features_df["label"].value_counts())

print("\nFailed reasons:")
display(failed_df.head())

Streaming output truncated to the last 5000 lines.
Extracting features:  45%|████▌     | 2059/4560 [07:22<09:00,  4.62it/s]/tmp/ipykernel_2284/2031551148.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  features["EDA_Phasic_AUC"] = np.trapz(phasic, dx=1 / fs)
Extracting features:  45%|████▌     | 2060/4560 [07:22<09:11,  4.53it/s]/tmp/ipykernel_2284/2031551148.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  features["EDA_Phasic_AUC"] = np.trapz(phasic, dx=1 / fs)
Extracting features:  45%|████▌     | 2061/4560 [07:22<08:54,  4.68it/s]/tmp/ipykernel_2284/2031551148.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  features["EDA_Phasic_AUC"] = np.trapz(phasic, dx=1 / fs)
Extracting features:  45%|████▌   

Valid feature rows: (4540, 137)
Failed rows: (20, 12)

Label counts:
label
REST      3902
STRESS     638
Name: count, dtype: int64

Failed reasons:


,recording_id,participant_id,block_id,stage_label,label,start_sec,end_sec,window_sec,bvp_error,eda_error,temp_error,acc_error
0,S10,S10,3,stroop,STRESS,390.0,420.0,30,low_ppg_quality,None,None,None
1,S10,S10,3,stroop,STRESS,405.0,435.0,30,low_ppg_quality,None,None,None
2,f10,f10,2,tmct,STRESS,338.0,368.0,30,too_few_ppg_peaks,None,None,None
3,f13,f13,3,post_tmct_rest,REST,1492.0,1522.0,30,None,too_many_eda_zeros,None,None
4,f13,f13,3,post_tmct_rest,REST,1507.0,1537.0,30,None,too_many_eda_zeros,None,None


## 8. Extraction Diagnostics and Quality Control

The following checks summarize failed feature-extraction windows, class and participant coverage, stage composition, baseline-normalization availability, and missing values before the final feature table is produced.

In [50]:
for col in ["bvp_error", "eda_error", "temp_error", "acc_error", "general_error"]:
    if col in failed_df.columns:
        print(f"\n{col}")
        print(failed_df[col].value_counts(dropna=True).head(20))


bvp_error
bvp_error
too_few_ppg_peaks    12
low_ppg_quality       6
Name: count, dtype: int64

eda_error
eda_error
too_many_eda_zeros    2
Name: count, dtype: int64

temp_error
Series([], Name: count, dtype: int64)

acc_error
Series([], Name: count, dtype: int64)


In [51]:
if not failed_df.empty:
    print("Failed rows by label:")
    print(failed_df["label"].value_counts())

    print("\nFailed rows by participant:")
    print(failed_df["participant_id"].value_counts().head(20))

Failed rows by label:
label
REST      17
STRESS     3
Name: count, dtype: int64

Failed rows by participant:
participant_id
f15    10
f13     5
S10     2
f18     2
f10     1
Name: count, dtype: int64


In [52]:
total_attempted = len(features_df) + len(failed_df)

print("Total attempted windows:", total_attempted)
print("Valid feature rows:", len(features_df))
print("Failed rows:", len(failed_df))
print("Failure rate:", round(len(failed_df) / total_attempted * 100, 2), "%")

print("\nValid label counts:")
print(features_df["label"].value_counts())

print("\nFailed label counts:")
print(failed_df["label"].value_counts())

Total attempted windows: 4560
Valid feature rows: 4540
Failed rows: 20
Failure rate: 0.44 %

Valid label counts:
label
REST      3902
STRESS     638
Name: count, dtype: int64

Failed label counts:
label
REST      17
STRESS     3
Name: count, dtype: int64


In [53]:
stage_counts = (
    features_df
    .groupby(["label", "stage_label"])
    .size()
    .reset_index(name="n_windows")
    .sort_values(["label", "n_windows"], ascending=[True, False])
)

display(stage_counts)

,label,stage_label,n_windows
4,REST,post_tmct_rest,1286
0,REST,baseline_rest,1275
1,REST,post_opposite_opinion_rest,863
3,REST,post_stroop_rest,354
5,REST,pre_task_rest,100
2,REST,post_real_opinion_rest,24
10,STRESS,tmct,431
8,STRESS,stroop,104
9,STRESS,subtraction,38
6,STRESS,opposite_opinion,33


In [54]:
stage_counts["percentage_within_label"] = (
    stage_counts["n_windows"] /
    stage_counts.groupby("label")["n_windows"].transform("sum") *
    100
)

display(stage_counts)

,label,stage_label,n_windows,percentage_within_label
4,REST,post_tmct_rest,1286,32.957458
0,REST,baseline_rest,1275,32.675551
1,REST,post_opposite_opinion_rest,863,22.116863
3,REST,post_stroop_rest,354,9.072271
5,REST,pre_task_rest,100,2.562788
2,REST,post_real_opinion_rest,24,0.615069
10,STRESS,tmct,431,67.554859
8,STRESS,stroop,104,16.300940
9,STRESS,subtraction,38,5.956113
6,STRESS,opposite_opinion,33,5.172414


In [55]:
participant_counts = (
    features_df
    .groupby(["participant_id", "label"])
    .size()
    .unstack(fill_value=0)
)

participant_counts["total"] = participant_counts.sum(axis=1)
participant_counts["stress_fraction"] = (
    participant_counts.get("STRESS", 0) / participant_counts["total"]
)

display(participant_counts.sort_values("stress_fraction"))

label,REST,STRESS,total,stress_fraction
participant_id,,,,
f12,267,18,285,0.063158
f05,257,19,276,0.068841
f13,195,16,211,0.075829
f03,191,18,209,0.086124
f15,221,21,242,0.086777
f04,181,19,200,0.095000
f18,157,17,174,0.097701
f16,163,18,181,0.099448
f06,154,18,172,0.104651


In [56]:
participants_missing_class = participant_counts[
    (participant_counts.get("REST", 0) == 0) |
    (participant_counts.get("STRESS", 0) == 0)
]

display(participants_missing_class)

label,REST,STRESS,total,stress_fraction
participant_id,,,,


In [57]:
features_df = features_df.copy()

features_df["is_baseline_for_norm"] = features_df["stage_label"].eq("baseline_rest")

# for f14, only f14_a is the true baseline
features_df.loc[
    (features_df["participant_id"] == "f14") &
    (features_df["recording_id"] != "f14_a"),
    "is_baseline_for_norm"
] = False

baseline_cols = [
    "TEMP_Mean",
    "EDA_Raw_Mean",
    "EDA_Tonic_Mean",
    "EDA_Phasic_Mean",
    "SCR_Peaks_N",
    "SCR_Amplitude_Mean",
    "PPG_Rate_Mean",
    "PPI_Mean",
    "HRV_RMSSD",
    "HRV_SDNN",
    "ACC_Mag_Mean",
]

baseline_cols = [col for col in baseline_cols if col in features_df.columns]

baseline_means = (
    features_df[features_df["is_baseline_for_norm"]]
    .groupby("participant_id")[baseline_cols]
    .mean()
    .add_suffix("_Baseline")
    .reset_index()
)

features_df = features_df.merge(
    baseline_means,
    on="participant_id",
    how="left"
)

for col in baseline_cols:
    features_df[f"{col}_DeltaBaseline"] = (
        features_df[col] - features_df[f"{col}_Baseline"]
    )

print("Added baseline-normalized delta features.")
features_df.head()

Added baseline-normalized delta features.


,recording_id,participant_id,block_id,stage_label,label,start_sec,end_sec,window_sec,BVP_Clean_Mean,BVP_Clean_Median,...,EDA_Raw_Mean_DeltaBaseline,EDA_Tonic_Mean_DeltaBaseline,EDA_Phasic_Mean_DeltaBaseline,SCR_Peaks_N_DeltaBaseline,SCR_Amplitude_Mean_DeltaBaseline,PPG_Rate_Mean_DeltaBaseline,PPI_Mean_DeltaBaseline,HRV_RMSSD_DeltaBaseline,HRV_SDNN_DeltaBaseline,ACC_Mag_Mean_DeltaBaseline
0,S01,S01,1,baseline_rest,REST,371.0,401.0,30,-0.031439,3.676886,...,-0.039180,-0.038771,-0.000162,8.0,-0.002414,1.044078,-0.006007,-40.023118,-76.694847,0.011893
1,S01,S01,1,baseline_rest,REST,386.0,416.0,30,0.253803,4.955127,...,-0.031460,-0.032067,0.000605,7.0,-0.002177,2.919855,-0.046908,-137.841593,-101.951098,0.014482
2,S01,S01,1,baseline_rest,REST,401.0,431.0,30,0.544310,3.131829,...,-0.020600,-0.019618,-0.001046,9.0,-0.002290,4.061531,-0.069623,-214.289657,-129.931664,0.015265
3,S01,S01,1,baseline_rest,REST,416.0,446.0,30,-0.237565,2.244995,...,0.003766,0.003907,-0.000089,-6.0,-0.001264,-4.899191,0.032047,-309.677131,-211.497203,0.015480
4,S01,S01,1,baseline_rest,REST,431.0,461.0,30,0.404348,1.102333,...,0.011091,0.011469,-0.000493,7.0,-0.003294,11.915276,-0.080226,-306.886828,-165.455071,0.010090


In [58]:
baseline_missing = features_df[
    [col for col in features_df.columns if col.endswith("_Baseline")]
].isnull().mean().sort_values(ascending=False)

display(baseline_missing.head(20))

,0
TEMP_Mean_Baseline,0.0
EDA_Raw_Mean_Baseline,0.0
EDA_Tonic_Mean_Baseline,0.0
EDA_Phasic_Mean_Baseline,0.0
SCR_Peaks_N_Baseline,0.0
SCR_Amplitude_Mean_Baseline,0.0
PPG_Rate_Mean_Baseline,0.0
PPI_Mean_Baseline,0.0
HRV_RMSSD_Baseline,0.0
HRV_SDNN_Baseline,0.0


In [59]:
features_df = features_df.replace([np.inf, -np.inf], np.nan)

missing_report = (
    features_df
    .isnull()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column", "missing_fraction"]

display(missing_report.head(40))

,column,missing_fraction
0,HRV_LFHF,1.000000
1,HRV_LF,1.000000
2,HRV_SampEn,0.224229
3,HRV_HF,0.000220
4,recording_id,0.000000
5,participant_id,0.000000
6,block_id,0.000000
7,stage_label,0.000000
8,BVP_Clean_Mean,0.000000
9,BVP_Clean_Median,0.000000


In [60]:
high_missing_cols = missing_report[
    missing_report["missing_fraction"] > 0.30
]["column"].tolist()

print("Columns with >30% missing:")
print(high_missing_cols)

Columns with >30% missing:
['HRV_LFHF', 'HRV_LF']


## 9. Final Cleaning and Export

An intermediate multimodal table and the failed-window log are saved first. Features with unacceptable missingness or instability are then removed before exporting the cleaned dataset consumed by Notebook 2.

In [61]:
features_df.to_csv(DATA_DIR / "stress_features_30s_multimodal.csv", index=False)
failed_df.to_csv(DATA_DIR / "stress_feature_extraction_failed_windows.csv", index=False)

In [62]:
drop_cols = [
    "HRV_LF",
    "HRV_LFHF",
    "HRV_SampEn"
]

drop_cols = [col for col in drop_cols if col in features_df.columns]

features_df = features_df.drop(columns=drop_cols)

print("Dropped columns:", drop_cols)
print("New shape:", features_df.shape)

Dropped columns: ['HRV_LF', 'HRV_LFHF', 'HRV_SampEn']
New shape: (4540, 157)


In [63]:
missing_report = (
    features_df
    .isnull()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column", "missing_fraction"]

display(missing_report.head(30))

,column,missing_fraction
0,HRV_HF,0.00022
1,participant_id,0.00000
2,block_id,0.00000
3,stage_label,0.00000
4,recording_id,0.00000
5,start_sec,0.00000
6,end_sec,0.00000
7,window_sec,0.00000
8,BVP_Clean_Mean,0.00000
9,BVP_Clean_Median,0.00000


In [64]:
features_df.to_csv(DATA_DIR / "stress_features_30s_multimodal_cleaned.csv", index=False)

In [65]:
print("Final dataset shape:", features_df.shape)

print("\nLabel counts:")
print(features_df["label"].value_counts())

print("\nLabel percentages:")
print((features_df["label"].value_counts(normalize=True) * 100).round(2))

print("\nNumber of participants:")
print(features_df["participant_id"].nunique())

print("\nNumber of recordings:")
print(features_df["recording_id"].nunique())

print("\nWindows per participant:")
display(
    features_df
    .groupby(["participant_id", "label"])
    .size()
    .unstack(fill_value=0)
)

Final dataset shape: (4540, 157)

Label counts:
label
REST      3902
STRESS     638
Name: count, dtype: int64

Label percentages:
label
REST      85.95
STRESS    14.05
Name: proportion, dtype: float64

Number of participants:
35

Number of recordings:
36

Windows per participant:


label,REST,STRESS
participant_id,,
S01,64,15
S02,48,16
S03,46,15
S04,59,16
S05,51,19
S06,52,17
S07,47,15
S08,45,18
S09,46,15
